## CSV Data Quality Audit

The goal of this file is to review the code in `Income_CalFairPlan_2021_2024.R` to ensure that the joins and cleaned CSV file are as expected. 

The CSV and R code were created by Nigel Yang. This data quality audit was created by Maya Schroder. The California Fair Plan data is from the [CFP website](https://www.cfpnet.com/key-statistics-data/) with PDFs converted to CSVs using Claude. The Census data is from [data.census.gov](https://data.census.gov/). The Census reports their data by ZCTA (ZIP code tabulation area), which is not the same as ZIP code. To convert everything to ZIP code, we used the CSV from [this Github repository](https://github.com/censusreporter/acs-aggregate/tree/master/crosswalks/zip_to_zcta).

The CSV includes the non-geographic data we are using for our analysis. It includes annual income and California Fair Plan enrollments by zip code from 2021 - 2024. (We expect to incorporate 2025 data after 10/29.)

In [1]:
# Import packages and raw data
from pathlib import Path
import pandas as pd

In [2]:
# file path to import data
root_fldr = Path.cwd().parent
raw_fldr = root_fldr / 'data' / 'raw'
processed_fldr = root_fldr / 'data' / 'processed'

# import Census data
raw_data_2024 = raw_fldr / 'ACS_2024_Income_Data.csv'
raw_data_2023 = raw_fldr / 'ACS_2023_Income_Data.csv'
raw_data_2022 = raw_fldr / 'ACS_2022_Income_Data.csv'
raw_data_2021 = raw_fldr / 'ACS_2021_Income_Data.csv'

census_2024 = pd.read_csv(raw_data_2024, skiprows=[1])
census_2023 = pd.read_csv(raw_data_2023, skiprows=[1])
census_2022 = pd.read_csv(raw_data_2022, skiprows=[1])
census_2021 = pd.read_csv(raw_data_2021, skiprows=[1])

# import Fair Plan data
raw_fp_data = raw_fldr / 'cfp_policies_by_zip.csv'
cfp_data = pd.read_csv(raw_fp_data)

# import ZCTA to zip code data
raw_zcta_zip = raw_fldr / 'zip_zcta_xref.csv'
zcta_to_zip = pd.read_csv(raw_zcta_zip)

### Step 1: Check income data by year and confirm that missing values look reasonable

In [3]:
census_2021.info()

# non-null count and number of entries are aligned, good sign

<class 'pandas.DataFrame'>
RangeIndex: 1808 entries, 0 to 1807
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   GEO_ID       1808 non-null   str    
 1   NAME         1808 non-null   str    
 2   B19013_001E  1808 non-null   str    
 3   B19013_001M  1808 non-null   str    
 4   Unnamed: 4   0 non-null      float64
dtypes: float64(1), str(4)
memory usage: 130.7 KB


In [6]:
census_2021['NAME'].isna().sum()

# no NA ZCTA values

np.int64(0)

In [7]:
census_2021 = census_2021.rename(columns={'NAME':'zcta'})

In [8]:
# repeat checks for each year

dfs = [census_2022, census_2023, census_2024]

missing = 0

for df in dfs:
    missing = missing + df['NAME'].isna().sum()

print(missing)

# no missing ZCTAs

0


### Step 2: Filter down zip code data to zip codes of interest

California ZIP codes fall between 90001 and 96162 (see [source](https://www.zipcode.com.ng/2022/06/list-of-california-zip-codes.html)).

In [9]:
zcta_to_zip.info()

<class 'pandas.DataFrame'>
RangeIndex: 41131 entries, 0 to 41130
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   zip_code  41131 non-null  int64  
 1   zcta      41125 non-null  float64
 2   source    41131 non-null  str    
dtypes: float64(1), int64(1), str(1)
memory usage: 1.3 MB


In [10]:
cali_zip_codes = zcta_to_zip.loc[zcta_to_zip['zip_code'].between(90001, 96162)]

cali_zip_codes.info()

<class 'pandas.DataFrame'>
Index: 2593 entries, 2317 to 40979
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   zip_code  2593 non-null   int64  
 1   zcta      2593 non-null   float64
 2   source    2593 non-null   str    
dtypes: float64(1), int64(1), str(1)
memory usage: 101.6 KB


In [11]:
# check number of distinct ZCTAs
cali_zip_codes['zcta'].unique() #1763

array([94501., 94502., 94536., ..., 95981., 95250., 95314.], shape=(1763,))

### Step 3: Compare ZCTAs in ACS to ZCTAs in cross-reference data

In [12]:
census_2021['zcta'] = census_2021['zcta'].str[5:].astype(float)

In [13]:
zcta_comp = pd.merge(
    census_2021, 
    cali_zip_codes,
    on='zcta',
    how='left',
    indicator=True
)

zcta_comp['_merge'].value_counts()

# 53 only in ACS data

_merge
both          2585
left_only       53
right_only       0
Name: count, dtype: int64

In [14]:
zcta_comp.loc[zcta_comp['_merge'] == 'left_only']

,GEO_ID,zcta,B19013_001E,B19013_001M,Unnamed: 4,zip_code,source,_merge
0,860Z200US89010,89010.0,36417,14295,NaN,NaN,NaN,left_only
1,860Z200US89019,89019.0,-,**,NaN,NaN,NaN,left_only
2,860Z200US89060,89060.0,51268,9640,NaN,NaN,NaN,left_only
3,860Z200US89061,89061.0,60848,4762,NaN,NaN,NaN,left_only
4,860Z200US89439,89439.0,70536,25160,NaN,NaN,NaN,left_only
79,860Z200US90052,90052.0,-,**,NaN,NaN,NaN,left_only
195,860Z200US90639,90639.0,-,**,NaN,NaN,NaN,left_only
277,860Z200US90840,90840.0,-,**,NaN,NaN,NaN,left_only
330,860Z200US91125,91125.0,-,**,NaN,NaN,NaN,left_only
642,860Z200US92092,92092.0,-,**,NaN,NaN,NaN,left_only


ZCTAs that start with 8 are not in California (e.g., 89010 is in Nevada). Other ZCTAs found in `left_only` include ones that have near zero population (e.g., 90052), which is why the income associated with these is typically null. For the five in California that have income data, they represent specialized circumstances that we do not need to capture in this project (e.g., 95671 represents Folsom State Prison in California).

#### Step 3.1: Confirm with other ACS years

In [ ]:
sum = 0

for df in dfs:
    df = df.rename(columns={'NAME':'zcta'})

    df['zcta'] = df['zcta'].str[5:].astype(float)

    zcta_comp = pd.merge(
    df, 
    cali_zip_codes,
    on='zcta',
    how='left',
    indicator=True
    )

    print(zcta_comp['_merge'].value_counts())

# expected value for all merges

_merge
both          2585
left_only       53
right_only       0
Name: count, dtype: int64
_merge
both          2585
left_only       53
right_only       0
Name: count, dtype: int64
_merge
both          2585
left_only       53
right_only       0
Name: count, dtype: int64


### Step 4: Pull in California Fair Plan data to compare with zip code/ZCTA data

In [23]:
cfp_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 1682 entries, 0 to 1681
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   zip            1682 non-null   int64
 1   policies_2025  1682 non-null   int64
 2   policies_2024  1682 non-null   int64
 3   policies_2023  1682 non-null   int64
 4   policies_2022  1682 non-null   int64
 5   policies_2021  1682 non-null   int64
dtypes: int64(6)
memory usage: 79.0 KB


In [ ]:
# join with zcta to zip code

cfp_data = cfp_data.rename(columns={
    'zip' : 'zip_code'
})

cfp_zips = pd.merge(
    cfp_data,
    zcta_to_zip,
    on='zip_code',
    how='left',
    indicator=True
)

cfp_zips['_merge'].value_counts()

# all zip codes align!

_merge
both          1682
left_only        0
right_only       0
Name: count, dtype: int64